In [ ]:
!npm install -g localtunnel


In [ ]:
import os
os.chdir('/content')
!rm -rf /content/venv
!pip install virtualenv
!virtualenv /content/venv
!/content/venv/bin/pip install fastapi uvicorn requests pydantic google-api-python-client "google-auth>=2,<3" google-auth-httplib2 google-auth-oauthlib
!rm -rf /content/usman-and-umar-app
!git clone --single-branch --branch feature/hybrid-task-orchestrator https://github.com/saedhawwa23-rgb/usman-and-umar-app.git /content/usman-and-umar-app
print("Fast backend packages and repository are ready.")


### Google authentication (standalone debug cell)

Run the next cell by itself to test Google credentials; it does not start the backend or tunnel. The default path sets `USE_AUTH_EPHEM=0` before Colab authentication to request the legacy credential flow. If that still fails, add a Colab Secret named `GOOGLE_SERVICE_ACCOUNT_JSON` containing the complete Service Account JSON key (enable notebook access), then rerun the cell. For this alternative, enable the Google Drive API and Cloud Firestore API in the Service Account project and share the checkpoint Drive folder with the Service Account email as Editor. Never paste the key into a notebook cell or commit it.


In [ ]:
import os
from pathlib import Path
from google.colab import auth, userdata
from google.auth.transport.requests import Request

DRIVE_SCOPE = "https://www.googleapis.com/auth/drive"
service_account_json = None
try:
    service_account_json = userdata.get("GOOGLE_SERVICE_ACCOUNT_JSON")
except Exception:
    pass  # Optional fallback secret; use interactive Colab auth below.

if service_account_json:
    key_path = Path("/content/colab-service-account.json")
    key_path.write_text(service_account_json)
    key_path.chmod(0o600)
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(key_path)
    from google.oauth2 import service_account
    credentials = service_account.Credentials.from_service_account_file(
        str(key_path), scopes=[DRIVE_SCOPE, "https://www.googleapis.com/auth/cloud-platform"]
    )
    os.environ["GOOGLE_CLOUD_PROJECT"] = credentials.project_id
    auth_mode = "service_account"
else:
    os.environ["USE_AUTH_EPHEM"] = "0"
    auth.authenticate_user()
    import google.auth
    credentials, _ = google.auth.default(scopes=[DRIVE_SCOPE, "https://www.googleapis.com/auth/cloud-platform"])
    auth_mode = "legacy_colab_oauth"

if not credentials.valid:
    credentials.refresh(Request())
print(f"Google authentication succeeded using {auth_mode}; credentials were not displayed.")


In [ ]:
# Verify Drive persistence before starting the API. Run the standalone Google authentication cell above first.
import hashlib, json, os, re, uuid
from datetime import datetime, timezone
from google.colab import userdata
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaIoBaseUpload
import io

# Secret values are copied only into this runtime's environment; never print their contents.
SECRET_NAMES = (
    "APP_CLIENT_TOKEN", "NVIDIA_API_KEY", "OPENROUTER_API_KEY", "OPENROUTER_MODEL",
    "GITHUB_TOKEN", "OPENSWARM_DRIVE_FOLDER_ID", "TUNNEL_SUBDOMAIN",
)
for _name in SECRET_NAMES:
    try:
        _value = userdata.get(_name)
    except Exception:
        _value = None
    if isinstance(_value, str) and _value.strip():
        os.environ[_name] = _value.strip()
    elif _name not in os.environ:
        os.environ.pop(_name, None)
print("Colab secrets loaded where available (values are hidden).")

# Each runtime gets new identities so a restarted Colab session cannot silently reuse an old instance ID.
_runtime_nonce = uuid.uuid4().hex
os.environ["OPENSWARM_SERVER_ID"] = f"colab-{_runtime_nonce[:12]}"
os.environ["OPENSWARM_INSTANCE_ID"] = _runtime_nonce

DRIVE_SCOPE = "https://www.googleapis.com/auth/drive"
DEFAULT_FOLDER_NAME = "USMAN_UMAR_CHECKPOINTS"
try:
    drive = build("drive", "v3", credentials=credentials, cache_discovery=False)

    configured_folder_id = os.environ.get("OPENSWARM_DRIVE_FOLDER_ID", "").strip()
    if configured_folder_id:
        folder = drive.files().get(
            fileId=configured_folder_id,
            fields="id,name,mimeType,capabilities(canAddChildren)",
        ).execute()
        if folder.get("mimeType") != "application/vnd.google-apps.folder":
            raise RuntimeError("OPENSWARM_DRIVE_FOLDER_ID does not identify a Drive folder.")
    else:
        escaped_name = DEFAULT_FOLDER_NAME.replace("'", "\'")
        query = (
            f"name = '{escaped_name}' and mimeType = 'application/vnd.google-apps.folder' "
            "and trashed = false"
        )
        response = drive.files().list(
            q=query, spaces="drive", pageSize=1000,
            fields="files(id,name,mimeType,capabilities(canAddChildren)),nextPageToken",
        ).execute()
        if response.get("nextPageToken"):
            raise RuntimeError("Drive returned more folder matches than can be checked safely.")
        matches = response.get("files", [])
        if len(matches) > 1:
            raise RuntimeError(
                f"Found {len(matches)} folders named {DEFAULT_FOLDER_NAME}; refusing ambiguous selection. "
                "Set OPENSWARM_DRIVE_FOLDER_ID in Colab Secrets to an explicit folder ID."
            )
        if len(matches) == 1:
            folder = matches[0]
        else:
            folder = drive.files().create(
                body={"name": DEFAULT_FOLDER_NAME, "mimeType": "application/vnd.google-apps.folder"},
                fields="id,name,mimeType,capabilities(canAddChildren)",
            ).execute()

    if folder.get("capabilities", {}).get("canAddChildren") is not True:
        # Re-read the capability explicitly so a missing value is not mistaken for access.
        folder = drive.files().get(
            fileId=folder["id"],
            fields="id,name,mimeType,capabilities(canAddChildren)",
        ).execute()
    if folder.get("capabilities", {}).get("canAddChildren") is not True:
        raise RuntimeError("The selected Drive folder is not writable by this authenticated account.")

    os.environ["OPENSWARM_DRIVE_FOLDER_ID"] = folder["id"]
    # Round-trip a uniquely named temporary JSON file in the target folder and remove it.
    probe_id = uuid.uuid4().hex
    probe_name = f".usman_drive_preflight_{probe_id}.json"
    probe_doc = {
        "purpose": "startup_preflight",
        "nonce": probe_id,
        "created_at": datetime.now(timezone.utc).isoformat(),
    }
    probe_bytes = json.dumps(probe_doc, sort_keys=True, separators=(",", ":")).encode("utf-8")
    probe_file_id = None
    probe_error = None
    try:
        created = drive.files().create(
            body={"name": probe_name, "mimeType": "application/json", "parents": [folder["id"]]},
            media_body=MediaIoBaseUpload(io.BytesIO(probe_bytes), mimetype="application/json", resumable=False),
            fields="id,name,parents",
        ).execute()
        probe_file_id = created.get("id")
        if not probe_file_id:
            raise RuntimeError("Drive did not return an ID for the temporary preflight file.")
        downloaded = drive.files().get_media(fileId=probe_file_id).execute()
        if not isinstance(downloaded, (bytes, bytearray)) or bytes(downloaded) != probe_bytes:
            raise RuntimeError("Drive preflight JSON read-back did not exactly match the written bytes.")
        if json.loads(bytes(downloaded).decode("utf-8")) != probe_doc:
            raise RuntimeError("Drive preflight JSON content verification failed.")
    except Exception as exc:
        probe_error = exc
    finally:
        if probe_file_id:
            try:
                drive.files().delete(fileId=probe_file_id).execute()
            except Exception as cleanup_exc:
                raise RuntimeError(
                    "Drive preflight temporary-file cleanup failed; backend will not be started."
                ) from cleanup_exc
    if probe_error:
        raise RuntimeError("Drive folder write/read-back verification failed; backend will not be started.") from probe_error

    print(f"Drive preflight passed for folder {folder.get('name', DEFAULT_FOLDER_NAME)!r}; temporary JSON was verified and removed.")
    print("Runtime server/instance IDs were set; token and model secret values remain hidden.")
except Exception as exc:
    print(f"NOT READY — Google authentication or Drive preflight failed: {type(exc).__name__}: {exc}")
    raise


In [ ]:
# Start FastAPI only after the authenticated Drive preflight has succeeded.
# The subprocess runs the same command as: uvicorn main:app --host 0.0.0.0 --port 7860.
import os, subprocess, sys, time
from pathlib import Path
backend_dir = Path('/content/usman-and-umar-app/cloud-backend')
if not backend_dir.is_dir():
    raise RuntimeError("Backend folder missing. Repository clone failed; server will not be started.")
os.chdir(backend_dir)
backend_log_path = Path('/content/fastapi.log')
backend_log = open(backend_log_path, 'w', buffering=1)
try:
    fastapi_process = subprocess.Popen(
        ['/content/venv/bin/python', '-m', 'uvicorn', 'main:app', '--host', '0.0.0.0', '--port', '7860'],
        cwd=str(backend_dir), stdout=backend_log, stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )
except Exception as exc:
    backend_log.flush()
    backend_log.close()
    print(f"FAILED to launch FastAPI/uvicorn: {type(exc).__name__}: {exc}")
    if backend_log_path.exists():
        print("FastAPI startup log (last 80 lines):")
        print('\n'.join(backend_log_path.read_text(errors='replace').splitlines()[-80:]) or '<empty>')
    raise RuntimeError("Could not start uvicorn on 0.0.0.0:7860; see the startup diagnostics above.") from exc
print(f"FastAPI process started (PID {fastapi_process.pid}) on 0.0.0.0:7860; log: {backend_log_path}")
print("Waiting for Drive/checkpoint/lease readiness.")


In [ ]:
# Require the backend's real readiness contract. A legacy {"status":"ok"} response is NOT ready.
import requests, time

health_url = 'http://127.0.0.1:7860/health'
required_true = ('ready', 'auth_ready', 'drive_ready', 'checkpoints_ready', 'checkpoint_storage_ready', 'lease_ready', 'lease_coordinator_ready', 'accepting_new_tasks')
required_false = ('draining',)
deadline = time.monotonic() + 180
last_health = None
health_ok = False
last_error = 'No health response received yet.'
while time.monotonic() < deadline:
    if fastapi_process.poll() is not None:
        last_error = f"FastAPI exited with code {fastapi_process.returncode}."
        try:
            backend_log.flush()
            print("FastAPI process exited; startup log (last 100 lines):")
            print('\n'.join(backend_log_path.read_text(errors='replace').splitlines()[-100:]) or '<empty>')
        except Exception as log_exc:
            print(f"Could not read FastAPI startup log: {type(log_exc).__name__}: {log_exc}")
        break
    try:
        response = requests.get(health_url, timeout=5)
        last_health = response.json() if response.headers.get('content-type', '').startswith('application/json') else None
        if response.status_code == 200 and isinstance(last_health, dict):
            missing_or_false = [key for key in required_true if last_health.get(key) is not True]
            missing_or_true = [key for key in required_false if last_health.get(key) is not False]
            if not missing_or_false and not missing_or_true:
                health_ok = True
                print("Backend health is ready: Drive, checkpoints, lease, auth, and accepting-new-tasks readiness are all true.")
                break
            last_error = f"Health has not declared readiness; false/missing checks: {missing_or_false}; true/missing drain flags: {missing_or_true}; response: {last_health}"
        else:
            last_error = f"Health returned HTTP {response.status_code}: {last_health or response.text[:500]}"
    except Exception as exc:
        last_error = f"Health request failed: {type(exc).__name__}: {exc}"
    time.sleep(3)
else:
    missing_or_false = [key for key in required_true if not isinstance(last_health, dict) or last_health.get(key) is not True]
    missing_or_true = [key for key in required_false if not isinstance(last_health, dict) or last_health.get(key) is not False]

if not (health_ok and isinstance(last_health, dict)
        and all(last_health.get(key) is True for key in required_true)
        and all(last_health.get(key) is False for key in required_false)):
    if fastapi_process.poll() is None:
        fastapi_process.terminate()
        try:
            fastapi_process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            fastapi_process.kill()
    print(f"NOT READY — tunnel will not be launched. Last health result: {last_error}")
    try:
        backend_log.flush()
        print("FastAPI log (last 100 lines):")
        print('\n'.join(backend_log_path.read_text(errors='replace').splitlines()[-100:]) or '<empty>')
    except Exception as log_exc:
        print(f"Could not read FastAPI log: {type(log_exc).__name__}: {log_exc}")
    raise RuntimeError("FastAPI did not report ready, auth_ready, drive_ready, checkpoints_ready, lease_ready, and accepting_new_tasks=true.")


### Configure a per-server Localtunnel subdomain

In Google Colab, open **Secrets** (key icon), add a secret named `TUNNEL_SUBDOMAIN`, enter a unique value such as `usman-umar-1` for Server 1 or `usman-umar-2` for Server 2, and enable notebook access. If unset, the tunnel cell defaults to `usman-umar-1`. The tunnel starts only after the health gate above reports fully ready.


In [ ]:
# Tunnel launch is deliberately after successful Drive/checkpoint/lease health readiness.
import os, re, subprocess, time
try:
    from google.colab import userdata
    _tunnel_secret = userdata.get("TUNNEL_SUBDOMAIN")
    if isinstance(_tunnel_secret, str) and _tunnel_secret.strip():
        os.environ["TUNNEL_SUBDOMAIN"] = _tunnel_secret.strip()
except Exception:
    pass  # Environment/default fallback; the secret itself is never printed.
subdomain = os.environ.get("TUNNEL_SUBDOMAIN", "usman-umar-1").strip()
if not re.fullmatch(r"[a-zA-Z0-9-]{1,63}", subdomain):
    raise RuntimeError("TUNNEL_SUBDOMAIN contains invalid characters; tunnel was not launched.")
tunnel_log = open('/content/localtunnel.log', 'w')
tunnel_process = subprocess.Popen(
    ['npx', '--yes', 'localtunnel', '--port', '7860', '--subdomain', subdomain],
    stdout=tunnel_log, stderr=subprocess.STDOUT, env=os.environ.copy(),
)
time.sleep(3)
if tunnel_process.poll() is not None:
    tunnel_log.flush()
    print("Localtunnel exited before becoming available; inspect /content/localtunnel.log. No URL is being published.")
    raise RuntimeError(f"Localtunnel exited with code {tunnel_process.returncode}.")
print(f"Public URL: https://{subdomain}.loca.lt")
print("Open the URL in Chrome first to bypass the Localtunnel warning page.")


In [ ]:
# Includes langchain-openai, required by OpenSwarm's ChatOpenAI model adapter.
import subprocess
_ai_log = open('/content/pip_ai.log', 'w')
ai_install_process = subprocess.Popen(
    ['/content/venv/bin/pip', 'install', '-r', '/content/usman-and-umar-app/cloud-backend/requirements-ai.txt'],
    stdout=_ai_log, stderr=subprocess.STDOUT,
)
print("Downloading OpenSwarm AI packages in the background. AI tasks will work once installation completes.")


In [ ]:
import importlib.util, subprocess, sys, time
ready = False
for _ in range(6):
    if ai_install_process.poll() not in (None, 0):
        print("OpenSwarm AI package installation failed; inspect /content/pip_ai.log.")
        raise RuntimeError(f"AI package installer exited with code {ai_install_process.returncode}.")
    check = subprocess.run(['/content/venv/bin/python', '-c', "import importlib.util,sys; sys.exit(0 if importlib.util.find_spec('langchain') else 1)"], check=False)
    if check.returncode == 0:
        print('OpenSwarm AI Engine is ready!')
        ready = True
        break
    time.sleep(5)
if not ready:
    print('OpenSwarm AI packages are still installing in the background; inspect /content/pip_ai.log if this persists.')


## Drain limitation

Colab does not provide this notebook with a dependable runtime shutdown/drain hook. Runtime expiration, disconnection, or forced termination can stop FastAPI without a graceful task drain; do not assume an in-flight task is checkpointed before shutdown. Drive recovery is conservative, best-effort advisory coordination—not exclusive ownership or guaranteed fencing. A confirmed readiness response and Drive read-back are not guarantees against concurrent writers or abrupt runtime loss.
